# Pelagic: in-process acoustics
Run from a repository checkout with `pip install 'pelagic[examples]'`. This notebook is executed in CI. The fixtures are pinned Acoustics Toolbox v2023.5 evidence, not newly generated Pelagic results.
The small comparisons below illustrate the fixed reference criteria; they do not replace the complete Rust numerical acceptance suite.

In [ ]:
from pathlib import Path
import struct
import numpy as np
import matplotlib.pyplot as plt
import pelagic as pg
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Cargo.toml').is_file())
BH = ROOT / 'crates/bellhop/tests/fixtures/golden'
KR = ROOT / 'crates/kraken/tests/fixtures'
print(pg.__version__, pg.compatibility_reference)

## A pressure field, with Fortran comparison
The public pressure axes are source, receiver depth, receiver range. The committed CSV is range-major. Transmission loss is **−20 log₁₀|p|**, not pressure level.

In [ ]:
case = pg.bellhop.load(BH / 'Field_G.env')
result = case.run()
reference = np.loadtxt(BH / 'Field_G.csv', delimiter=',', skiprows=1)
flat = result.field.pressure.transpose(0, 2, 1).ravel()
np.testing.assert_allclose(flat.real, reference[:, 2], rtol=0, atol=5e-8)
np.testing.assert_allclose(flat.imag, reference[:, 3], rtol=0, atol=5e-8)
plt.pcolormesh(result.field.ranges_m, result.field.depths_m, result.field.transmission_loss()[0], shading='auto')
plt.gca().invert_yaxis(); plt.colorbar(label='TL (dB)'); plt.xlabel('Range (m)'); plt.ylabel('Depth (m)');
print('Maximum component error:', max(abs(flat.real-reference[:, 2]).max(), abs(flat.imag-reference[:, 3]).max()))

## Munk: one ray and its pinned Fortran trajectory
Trajectory columns are range, depth, travel time, attenuation time, amplitude and phase. Metadata carries launch angle, bounce counts and termination separately.

In [ ]:
munk = pg.bellhop.load(BH / 'MunkP_one_ray.env').run()
ray = munk.rays[0]
reference_ray = np.loadtxt(BH / 'MunkP_one_ray.ray', skiprows=9)
np.testing.assert_allclose(ray.points[:, :2], reference_ray, rtol=0, atol=1e-5)
plt.figure(); plt.plot(ray.points[:, 0]/1000, ray.points[:, 1], label='Pelagic')
plt.plot(reference_ray[:, 0]/1000, reference_ray[:, 1], '.', label='Fortran v2023.5');
plt.gca().invert_yaxis(); plt.xlabel('Range (km)'); plt.ylabel('Depth (m)'); plt.legend();

## Pekeris: modes and FIELD, not modes-only
KRAKEN loads ENV plus FLP. Frequencies remain ordered; each profile has its own modal arrays. The following small reader is restricted to this single-frequency, single-profile pinned MOD fixture; production readers and full shape/pressure checks live in the Rust tests.

In [ ]:
pekeris = pg.kraken.load(KR / 'Pekeris.env').run()
modes = pekeris.frequencies[0].profiles[0].modes
raw = (KR / 'golden/Pekeris.mod').read_bytes()
record_bytes = 4 * struct.unpack_from('<i', raw)[0]
count = struct.unpack_from('<i', raw, 5*record_bytes)[0]
assert count == len(modes.wavenumbers)
blocks = [np.frombuffer(raw, dtype='<c8', count=min(record_bytes//8, count-start), offset=(7+count+start//(record_bytes//8))*record_bytes) for start in range(0, count, record_bytes//8)]
reference_k = np.concatenate(blocks)
np.testing.assert_array_equal(modes.wavenumbers.real.astype(np.float32), reference_k.real)
np.testing.assert_allclose(modes.wavenumbers.imag, reference_k.imag, rtol=0, atol=1e-10)
plt.figure(); plt.plot(modes.eigenfunctions.real[:, :3], modes.depths_m);
plt.gca().invert_yaxis(); plt.xlabel('Reference-normalized mode shape'); plt.ylabel('Depth (m)');

## Gulf: portable auxiliary inputs
This example consumes its SSP and bathymetry only when loading. Its exported JSON is self-contained. The equality below checks portability, **not independent Fortran parity**; Gulf fresh-reference comparison is provided by `tools/reference/compare-ray.sh crates/bellhop/tests/fixtures/Gulf_ray_rd.env` and requires the pinned Docker reference.

In [ ]:
gulf = pg.bellhop.load(ROOT / 'crates/bellhop/tests/fixtures/Gulf_ray_rd.env')
portable = pg.bellhop.Case.from_dict(gulf.to_dict())
a, b = gulf.run(), portable.run()
assert len(a.rays) == len(b.rays)
for left, right in zip(a.rays, b.rays): np.testing.assert_array_equal(left.points, right.points)
plt.figure()
for ray in a.rays: plt.plot(ray.points[:, 0]/1000, ray.points[:, 1])
plt.gca().invert_yaxis(); plt.xlabel('Range (km)'); plt.ylabel('Depth (m)');
print(a.provenance)